## Referring Columns using Spark Data Frame Names

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Referring Columns using Spark Data Frame Names").getOrCreate()

24/02/22 20:02:07 WARN Utils: Your hostname, Qasim resolves to a loopback address: 127.0.1.1; using 172.30.54.131 instead (on interface eth0)
24/02/22 20:02:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/02/22 20:02:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/02/22 20:02:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/02/22 20:02:10 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
24/02/22 20:02:10 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
24/02/22 20:02:10 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
24/02/22 20:02:10 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
24

In [2]:
import datetime
users_list = [
    {
        'id': 1,
        'first_name': 'John',
        'last_name': 'Doe',
        'email': 'john.doe@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 54 7400113', 'PAK_Mobile': '+92 300 5638080'},
        'is_customer': True,
        'amount_paid': 123.45,
        'customer_from': datetime.date(2023, 1, 15),  # Date in YYYY-MM-DD format
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)  # Timestamp in YYYY-MM-DD HH:MM:SS format
    },
    {
        'id': 2,
        'first_name': 'Jane',
        'last_name': 'Smith',
        'email': 'jane.smith@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 99 7666666', 'PAK_Mobile': '+92 300 0000000'},
        'is_customer': True,
        'amount_paid': 0.0,
        'customer_from': datetime.date(2022, 11, 10),
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)
    },
    {
        'id': 3,
        'first_name': 'Jack',
        'last_name': 'Alpha',
        'email': 'jack.alpha@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 99 7666555'},
        'is_customer': False,
        'amount_paid': 0.0,
        'customer_from': datetime.date(2022, 11, 10),
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)
    }
    # Add more user dictionaries as needed
]

In [3]:
from pyspark.sql import Row

users_df = spark.createDataFrame(Row(**users) for users in users_list)
users_df

DataFrame[id: bigint, first_name: string, last_name: string, email: string, phone_numer: map<string,string>, is_customer: boolean, amount_paid: double, customer_from: date, last_updated_ts: timestamp]

In [5]:
users_df['id']

Column<'id'>

In [6]:
users_df['id'].show()

TypeError: 'Column' object is not callable

In [7]:
from pyspark.sql.functions import col

In [8]:
col('id')

Column<'id'>

In [12]:
# users_df.select(['id'], col('first_name'), 'last_name') returns error

users_df.select('id', col('first_name'), 'last_name')

DataFrame[id: bigint, first_name: string, last_name: string]

In [13]:
users_df.select(users_df['id'], col('first_name'), 'last_name')

DataFrame[id: bigint, first_name: string, last_name: string]

## NOTE: selectExpr only take strings or arrays, not col type objects

In [18]:
# users_df.alias('ud').selectExpr(users_df['id'], 'first_name')

# this will give error since users_df['id'] is not iterable

In [19]:
users_df.createOrReplaceTempView('users_vw')

In [21]:
spark.sql("""
    SELECT u.id, first_name, last_name, concat(u.first_name, ', ', last_name) 
    from users_vw as u
"""
    ). \
    show()

+---+----------+---------+---------------------------------+
| id|first_name|last_name|concat(first_name, , , last_name)|
+---+----------+---------+---------------------------------+
|  1|      John|      Doe|                        John, Doe|
|  2|      Jane|    Smith|                      Jane, Smith|
|  3|      Jack|    Alpha|                      Jack, Alpha|
+---+----------+---------+---------------------------------+

